In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

# Base folder for all input files
base_dir = Path("data")
buildings_dir = base_dir / "buildings"
roads_dir = base_dir / "roads"
output_dir = base_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# Buildings
buildings_list = []
for shp_file in buildings_dir.glob("*.shp"):
    buildings_list.append(gpd.read_file(shp_file))
merged_buildings = pd.concat(buildings_list, ignore_index=True)

# Roads
roads_list = []
for shp_file in roads_dir.glob("*.shp"):
    roads_list.append(gpd.read_file(shp_file))
merged_roads = pd.concat(roads_list, ignore_index=True)

# State/county layer
state_counties = gpd.read_file(base_dir / "your_state_counties.shp")

# Filter to state
state_boundary = state_counties[state_counties["STATE_NAME"] == "Your State Name"]

# Clip
clipped_buildings = gpd.overlay(merged_buildings, state_boundary, how="intersection")

# Spatial join
buildings_with_county = gpd.sjoin(
    clipped_buildings,
    state_counties,
    how="left",
    predicate="intersects"
)

# Dissolve
buildings_by_county = buildings_with_county.dissolve(by="COUNTY_NAME", aggfunc="sum")

# Output
output_dir = base_dir / "output"
output_dir.mkdir(exist_ok=True)
buildings_by_county.to_file(output_dir / "Buildings_By_County.shp")